# DropoutIA — Exploration et nettoyage des données (Niveau de risque)

Ce notebook nettoie le **deuxième jeu de données**, celui du niveau de risque
initial (`id_niveauRisqueInit`, 4 catégories : 1 = faible risque à 4 = risque élevé).



## 1. Chargement des données brutes

In [15]:
import pandas as pd
import numpy as np
import sys
sys.path.append("../src")
import nettoyage


df = pd.read_excel("../data/raw/decrochage (1).xlsx")
print(df.shape)
df.head()

(10000, 12)


,codeEleve,dateNaisEleve,id_Genre,Nefstat,NbrAbsence,NbrRedoublement,MoyenneNotes,CD_REG,CD_PROV,CD_MIL,CD_ETAB,id_niveauRisqueInit
0,A130013657,2006-05-04 00:00:00.000,1,2A33101010,NaN,2,8.188333,12,391,1,26032V,3
1,A130016119,2007-03-23 00:00:00.000,1,2A33101010,NaN,1,8.811667,12,391,1,13533J,2
2,A147003080,2007-06-17 00:00:00.000,1,2A33101010,NaN,2,10.228333,12,391,1,13528D,3
3,A134015543,2007-09-13 00:00:00.000,2,2A33101010,NaN,2,10.595000,12,391,1,24930X,3
4,A137019596,2007-05-07 00:00:00.000,1,2A33101010,NaN,2,8.970000,12,391,1,13533J,3


## 2. Premier coup d'œil : structure et types

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   codeEleve            10000 non-null  str    
 1   dateNaisEleve        10000 non-null  str    
 2   id_Genre             10000 non-null  int64  
 3   Nefstat              10000 non-null  str    
 4   NbrAbsence           4919 non-null   float64
 5   NbrRedoublement      10000 non-null  int64  
 6   MoyenneNotes         10000 non-null  float64
 7   CD_REG               10000 non-null  int64  
 8   CD_PROV              10000 non-null  int64  
 9   CD_MIL               10000 non-null  int64  
 10  CD_ETAB              10000 non-null  str    
 11  id_niveauRisqueInit  10000 non-null  int64  
dtypes: float64(2), int64(6), str(4)
memory usage: 937.6 KB


In [17]:
df.describe(include='all')

,codeEleve,dateNaisEleve,id_Genre,Nefstat,NbrAbsence,NbrRedoublement,MoyenneNotes,CD_REG,CD_PROV,CD_MIL,CD_ETAB,id_niveauRisqueInit
count,10000,10000,10000.00000,10000,4919.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000,10000.000000
unique,10000,1574,NaN,17,NaN,NaN,NaN,NaN,NaN,NaN,719,NaN
top,A130013657,2010-06-01 00:00:00.000,NaN,2A32101110,NaN,NaN,NaN,NaN,NaN,NaN,24930X,NaN
freq,1,37,NaN,3057,NaN,NaN,NaN,NaN,NaN,NaN,593,NaN
mean,NaN,NaN,1.49080,NaN,3.085790,0.584500,11.736564,4.451700,418.915200,1.183000,NaN,1.434200
std,NaN,NaN,0.49994,NaN,5.564729,0.801825,3.054478,4.955454,136.234146,0.386686,NaN,0.764937
min,NaN,NaN,1.00000,NaN,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,NaN,1.000000
25%,NaN,NaN,1.00000,NaN,0.000000,0.000000,9.694583,1.000000,391.000000,1.000000,NaN,1.000000
50%,NaN,NaN,1.00000,NaN,1.000000,0.000000,11.487500,1.000000,405.000000,1.000000,NaN,1.000000
75%,NaN,NaN,2.00000,NaN,4.000000,1.000000,13.738750,12.000000,511.000000,1.000000,NaN,2.000000


## 3. Valeurs manquantes

In [18]:
df.isnull().sum().sort_values(ascending=False)

NbrAbsence             5081
codeEleve                 0
id_Genre                  0
dateNaisEleve             0
Nefstat                   0
NbrRedoublement           0
MoyenneNotes              0
CD_REG                    0
CD_PROV                   0
CD_MIL                    0
CD_ETAB                   0
id_niveauRisqueInit       0
dtype: int64

## 4. Distribution de la cible : `id_niveauRisqueInit`

Contrairement au premier fichier (2 catégories), celui-ci a **4 catégories**
(1 = risque faible, 4 = risque élevé). Regardez bien le déséquilibre : les
niveaux 3 et 4 (les plus critiques) sont largement minoritaires — à garder
en tête pour la modélisation (Semaine 3).

In [19]:
df['id_niveauRisqueInit'].value_counts().sort_index()

id_niveauRisqueInit
1    7051
2    1860
3     785
4     304
Name: count, dtype: int64

In [20]:
# 1. Corrélations entre variables numériques
df.corr(numeric_only=True)

,id_Genre,NbrAbsence,NbrRedoublement,MoyenneNotes,CD_REG,CD_PROV,CD_MIL,id_niveauRisqueInit
id_Genre,1.000000,-0.172187,-0.170330,0.215428,-0.048157,-0.007468,0.010779,-0.173400
NbrAbsence,-0.172187,1.000000,0.365815,-0.300311,0.261281,-0.048241,-0.061260,0.432294
NbrRedoublement,-0.170330,0.365815,1.000000,-0.437206,0.449654,-0.116360,-0.080521,0.864052
MoyenneNotes,0.215428,-0.300311,-0.437206,1.000000,-0.163171,0.022524,0.037531,-0.501716
CD_REG,-0.048157,0.261281,0.449654,-0.163171,1.000000,-0.219913,-0.250239,0.447617
CD_PROV,-0.007468,-0.048241,-0.116360,0.022524,-0.219913,1.000000,-0.275651,-0.080318
CD_MIL,0.010779,-0.061260,-0.080521,0.037531,-0.250239,-0.275651,1.000000,-0.113803
id_niveauRisqueInit,-0.173400,0.432294,0.864052,-0.501716,0.447617,-0.080318,-0.113803,1.000000


## 5. Vérification : le manque de NbrAbsence est-il aléatoire ?

Même vérification que pour le premier fichier : est-ce que le fait d'avoir
NbrAbsence manquant dépend du niveau de risque ? Si oui, ça confirme qu'il
faut imputer (médiane + indicateur) plutôt que supprimer les lignes.

In [21]:
df.groupby('id_niveauRisqueInit')['NbrAbsence'].apply(lambda x: round(x.isnull().mean()*100, 1))

id_niveauRisqueInit
1    53.9
2    43.5
3    41.7
4    47.7
Name: NbrAbsence, dtype: float64

## 6. Nettoyage des données

### 6.1 Gestion de `NbrAbsence` manquant (médiane + colonne indicatrice)

In [22]:
df = nettoyage.gerer_absences_manquantes(df, colonne="NbrAbsence")
df[["NbrAbsence", "NbrAbsence_manquant"]].head(10)

,NbrAbsence,NbrAbsence_manquant
0,1.0,1
1,1.0,1
2,1.0,1
3,1.0,1
4,1.0,1
5,1.0,1
6,0.0,0
7,1.0,1
8,17.0,0
9,17.0,0


### 6.2 Transformer la date de naissance en âge

In [23]:
df = nettoyage.calculer_age(df, colonne_date="dateNaisEleve")
df[["dateNaisEleve", "age"]].head()

,dateNaisEleve,age
0,2006-05-04,18
1,2007-03-23,17
2,2007-06-17,17
3,2007-09-13,16
4,2007-05-07,17


In [24]:
# 2. Différence entre décrocheurs et non-décrocheurs
df.groupby('id_niveauRisqueInit')[['NbrAbsence', 'NbrRedoublement', 'MoyenneNotes', 'age']].mean()

,NbrAbsence,NbrRedoublement,MoyenneNotes,age
id_niveauRisqueInit,,,,
1,1.182385,0.216849,12.873946,13.849525
2,3.547849,0.942473,8.862825,14.101613
3,4.765605,2.044586,9.637962,15.887898
4,5.207237,3.151316,8.357856,16.838816


### 6.3 Encoder les colonnes catégorielles

⚠️ Attention : dans ce fichier, la colonne s'appelle `Nefstat` (majuscule),
contrairement à l'autre fichier où c'était `nefstat` (minuscule). Vérifiez
toujours le nom exact avec `df.columns` avant d'exécuter cette étape sur un
nouveau fichier.

In [25]:
colonnes_categorielles = ["Nefstat"]
df = nettoyage.encoder_categorielles(df, colonnes_categorielles)
df.head()

,codeEleve,dateNaisEleve,id_Genre,Nefstat,NbrAbsence,NbrRedoublement,MoyenneNotes,CD_REG,CD_PROV,CD_MIL,CD_ETAB,id_niveauRisqueInit,NbrAbsence_manquant,age
0,A130013657,2006-05-04,1,14,1.0,2,8.188333,12,391,1,26032V,3,1,18
1,A130016119,2007-03-23,1,14,1.0,1,8.811667,12,391,1,13533J,2,1,17
2,A147003080,2007-06-17,1,14,1.0,2,10.228333,12,391,1,13528D,3,1,17
3,A134015543,2007-09-13,2,14,1.0,2,10.595000,12,391,1,24930X,3,1,16
4,A137019596,2007-05-07,1,14,1.0,2,8.970000,12,391,1,13533J,3,1,17


### 6.4 Retirer les colonnes inutiles

On retire les identifiants (`codeEleve`, `dateNaisEleve` déjà transformée en âge,
`CD_ETAB`), ainsi que `CD_REG` et `CD_MIL` si elles n'ont qu'une seule valeur
dans ce fichier aussi (vérifiez avec la cellule suivante avant de les retirer).

In [26]:
print("CD_REG valeurs uniques :", df['CD_REG'].unique())
print("CD_MIL valeurs uniques :", df['CD_MIL'].unique())

CD_REG valeurs uniques : [12  5 10  9  6  7  4 11  1  8  3  2]
CD_MIL valeurs uniques : [1 2]


In [27]:
# Ajustez cette liste selon ce que la cellule précédente a montré :
# si CD_REG ou CD_MIL n'ont qu'une seule valeur, ajoutez-les à la liste ci-dessous
colonnes_a_retirer = ["codeEleve", "dateNaisEleve", "CD_ETAB"]
df_clean = nettoyage.retirer_identifiants(df, colonnes_a_retirer)
df_clean.head()

,id_Genre,Nefstat,NbrAbsence,NbrRedoublement,MoyenneNotes,CD_REG,CD_PROV,CD_MIL,id_niveauRisqueInit,NbrAbsence_manquant,age
0,1,14,1.0,2,8.188333,12,391,1,3,1,18
1,1,14,1.0,1,8.811667,12,391,1,2,1,17
2,1,14,1.0,2,10.228333,12,391,1,3,1,17
3,2,14,1.0,2,10.595000,12,391,1,3,1,16
4,1,14,1.0,2,8.970000,12,391,1,3,1,17


## 7. Sauvegarde des données nettoyées

In [28]:
df_clean.to_csv("../data/processed/donnees_risque_clean.csv", index=False)
print("Fichier sauvegardé :", df_clean.shape)

Fichier sauvegardé : (10000, 11)
